### TextLoaders

In [45]:
from dotenv import load_dotenv
load_dotenv()

True

### Get the documents

In [19]:
from pathlib import Path
import arxiv

# Search arXiv using a simple keyword.
query = "retrieval augmented generation"
number_of_papers = 5

# Create the folder automatically.
data_folder = Path("data/text_files")
data_folder.mkdir(parents=True, exist_ok=True)

# Ask arXiv for several papers matching the query.
search = arxiv.Search(
    query=query,
    max_results=number_of_papers,
    sort_by=arxiv.SortCriterion.Relevance,
)
client = arxiv.Client(page_size=number_of_papers)

# Store the paths of the text files we create.
text_paths = []

# Get one paper at a time, starting the file number at 1.
for file_number, paper in enumerate(client.results(search), start=1):
    text_path = data_folder / f"paper_{file_number:02d}.txt"
    paper_text = f"Title: {paper.title}\n\nAbstract:\n{paper.summary}\n"

    # Save the title and abstract as a local text file.
    text_path.write_text(paper_text, encoding="utf-8")
    text_paths.append(text_path)
    print(f"Saved: {text_path} - {paper.title}")

print(f"Created {len(text_paths)} text files.")

Saved: data\text_files\paper_01.txt - AR-RAG: Autoregressive Retrieval Augmentation for Image Generation
Saved: data\text_files\paper_02.txt - Intelligent Interaction Strategies for Context-Aware Cognitive Augmentation
Saved: data\text_files\paper_03.txt - Factually: Exploring Wearable Fact-Checking for Augmented Truth Discernment
Saved: data\text_files\paper_04.txt - Designing AI Systems that Augment Human Performed vs. Demonstrated Critical Thinking
Saved: data\text_files\paper_05.txt - Automated Literature Review Using NLP Techniques and LLM-Based Retrieval-Augmented Generation
Created 5 text files.


### Load the fetched documents

In [20]:
# TextLoader reads a plain-text file and creates a LangChain Document.
from langchain_community.document_loaders import TextLoader

# This list will hold all Document objects loaded from all text files.
documents = []

# Go through each file path created by the previous cell.
for text_path in text_paths:
    # Create a loader for the current file.
    loader = TextLoader(str(text_path), encoding="utf-8")

    # load() reads the file and returns a list of Document objects.
    # extend() adds those objects to our documents list.
    doc = loader.load()
    documents.extend(doc)

# There should be one Document for each text file.
print(f"Loaded {len(documents)} documents")

# page_content contains the actual text from the file.
print(documents[0].page_content[:200])

# metadata contains extra information, including the source file path.
print(documents[0].metadata)

Loaded 5 documents
Title: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation

Abstract:
We introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by aut
{'source': 'data\\text_files\\paper_01.txt'}


In [22]:
documents[0]

Document(metadata={'source': 'data\\text_files\\paper_01.txt'}, page_content='Title: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation\n\nAbstract:\nWe introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior methods that perform a single, static retrieval before generation and condition the entire generation on fixed reference images, AR-RAG performs context-aware retrievals at each generation step, using prior-generated patches as queries to retrieve and incorporate the most relevant patch-level visual references, enabling the model to respond to evolving generation needs while avoiding limitations (e.g., over-copying, stylistic bias, etc.) prevalent in existing methods. To realize AR-RAG, we propose two parallel frameworks: (1) Distribution-Augmentation in Decoding (DAiD), a training-free plug-and-use decoding strategy that

In [23]:
# Updating the metadata
documents[0].metadata["updated"]=True
documents

[Document(metadata={'source': 'data\\text_files\\paper_01.txt', 'updated': True}, page_content='Title: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation\n\nAbstract:\nWe introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior methods that perform a single, static retrieval before generation and condition the entire generation on fixed reference images, AR-RAG performs context-aware retrievals at each generation step, using prior-generated patches as queries to retrieve and incorporate the most relevant patch-level visual references, enabling the model to respond to evolving generation needs while avoiding limitations (e.g., over-copying, stylistic bias, etc.) prevalent in existing methods. To realize AR-RAG, we propose two parallel frameworks: (1) Distribution-Augmentation in Decoding (DAiD), a training-free plug-and-use deco

In [39]:
# Start with every document marked as not updated.
for document in documents:
    document.metadata["updated"] = True
documents

[Document(metadata={'source': 'data\\text_files\\paper_01.txt', 'updated': True}, page_content='Title: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation\n\nAbstract:\nWe introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior methods that perform a single, static retrieval before generation and condition the entire generation on fixed reference images, AR-RAG performs context-aware retrievals at each generation step, using prior-generated patches as queries to retrieve and incorporate the most relevant patch-level visual references, enabling the model to respond to evolving generation needs while avoiding limitations (e.g., over-copying, stylistic bias, etc.) prevalent in existing methods. To realize AR-RAG, we propose two parallel frameworks: (1) Distribution-Augmentation in Decoding (DAiD), a training-free plug-and-use deco

In [40]:
# This generator updates one document, then pauses at yield.
def update_metadata(documents):
    for document in documents:
        document.metadata["updated"] = False
        yield document


# Creating the generator does not run the function yet.
updated_documents = update_metadata(documents)

# next() runs only until the first yield.
first_document = next(updated_documents)
print("First document returned by next():")
print(first_document.metadata)

# The generator is paused. The remaining documents were not updated.
print("All documents after one next() call:")
documents

First document returned by next():
{'source': 'data\\text_files\\paper_01.txt', 'updated': False}
All documents after one next() call:


[Document(metadata={'source': 'data\\text_files\\paper_01.txt', 'updated': False}, page_content='Title: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation\n\nAbstract:\nWe introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior methods that perform a single, static retrieval before generation and condition the entire generation on fixed reference images, AR-RAG performs context-aware retrievals at each generation step, using prior-generated patches as queries to retrieve and incorporate the most relevant patch-level visual references, enabling the model to respond to evolving generation needs while avoiding limitations (e.g., over-copying, stylistic bias, etc.) prevalent in existing methods. To realize AR-RAG, we propose two parallel frameworks: (1) Distribution-Augmentation in Decoding (DAiD), a training-free plug-and-use dec

In [41]:
second_document = next(updated_documents)
documents

[Document(metadata={'source': 'data\\text_files\\paper_01.txt', 'updated': False}, page_content='Title: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation\n\nAbstract:\nWe introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior methods that perform a single, static retrieval before generation and condition the entire generation on fixed reference images, AR-RAG performs context-aware retrievals at each generation step, using prior-generated patches as queries to retrieve and incorporate the most relevant patch-level visual references, enabling the model to respond to evolving generation needs while avoiding limitations (e.g., over-copying, stylistic bias, etc.) prevalent in existing methods. To realize AR-RAG, we propose two parallel frameworks: (1) Distribution-Augmentation in Decoding (DAiD), a training-free plug-and-use dec

### Yield + next() combination

In [42]:
# 1. Define a generator function that splits a sentence into words
def word_stream(sentence: str):
    words = sentence.split(" ")  # Creates ['I', 'love', 'programming']
    for word in words:
        yield word               # Yields one word at a time and pauses

# 2. Instantiate the generator with our string
stream = word_stream("I love programming")

# 3. Retrieve words individually using next()
print(next(stream))  # Output: "I"           (Pauses after first word)

I


In [43]:
print(next(stream))  # Output: "love"        (Resumes and pauses after second word)

love


In [44]:
print(next(stream))  # Output: "programming" (Resumes and pauses after third word)

programming
